# 🧠 What I Want to Build
---

A chatbot that:

1. Takes a reference website (Liquid docs)
2. Converts it into a knowledge base
3. Answers questions only from that knowledge
4. If answer not found → responds:
    >“I don't have an answer for it”

## 🧱 Step-by-Step Implementation

#### 1. 📥 Ingest Liquid Documentation & Save Data

From this root:
👉 https://shopify.dev/docs/api/liquid

1. Visit the page
2. Extract all internal links
3. Visit each link
4. Avoid duplicates
5. Extract clean text
6. Store everything for chunking

In [48]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time

visited = set()
to_visit = set()

BASE_URL = "https://shopify.dev/docs/api/liquid"
DOMAIN = "shopify.dev"

to_visit.add(BASE_URL)

In [49]:
# Clean URLs
def clean_url(url):
    parsed = urlparse(url)
    return parsed.scheme + "://" + parsed.netloc + parsed.path

In [50]:
# Extract Only Useful Content
def extract_content(soup):
    main_content = soup.find("main")
    
    if main_content:
        text = main_content.get_text(separator=" ", strip=True)
    else:
        text = soup.get_text(separator=" ", strip=True)
    return text

In [51]:
# Crawl Loop
def crawl(max_pages=50):
    all_text = []

    while to_visit and len(visited) < max_pages:
        url = clean_url(to_visit.pop())

        if url in visited:
            continue

        print(f"Crawling: {url}")
        visited.add(url)

        try:
            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.text, "html.parser")
            # Extract text
            text = extract_content(soup)
            # store data
            all_text.append({
             "url": url,
             "content": text
            })

            # Extract links
            for link in soup.find_all("a", href=True):
                href = link["href"]
                full_url = clean_url(urljoin(url, href))

                # Keep only internal links
                if DOMAIN in urlparse(full_url).netloc:
                    if full_url not in visited:
                        to_visit.add(full_url)

            time.sleep(1)  # polite crawling

        except Exception as e:
            print(f"Error: {e}")

    return all_text

In [52]:
# save data
def save_data(docs, path="data/liquid_docs.json"):
    import os
    import json

    os.makedirs(os.path.dirname(path), exist_ok=True)

    with open(path, "w") as f:
        json.dump(docs, f)

    print(f"Saved {len(docs)} documents")


In [53]:
# Verify crawled data
docs = crawl(10)

print(f"\nTotal pages scraped: {len(docs)}")

# dump the data to avoid crawling everytime
save_data(docs)

Crawling: https://shopify.dev/docs/api/liquid

Total pages scraped: 1
Saved 1 documents


#### 2: Chunk + Embed + FAISS

In [54]:
import json
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load data
with open("data/liquid_docs.json") as f:
    docs = json.load(f)


# 🔪 Chunking
def chunk_text(text, size=500, overlap=50):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunks.append(text[i:i+size])
    return chunks


all_chunks = []
metadata = []

for doc in docs:
    chunks = chunk_text(doc["content"])
    for chunk in chunks:
        all_chunks.append(chunk)
        metadata.append(doc["url"])


# 🧠 Embedding
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(all_chunks)


# ⚡ FAISS Index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# 💾 Save everything
faiss.write_index(index, "data/index.faiss")

with open("data/chunks.json", "w") as f:
    json.dump({
        "chunks": all_chunks,
        "metadata": metadata
    }, f)

print("Embedding + Indexing complete")


Embedding + Indexing complete


#### 3: Chatbot (RAG + Guardrails)

In [58]:
# chatbot.py

import json
import faiss
import numpy as np
import requests
from sentence_transformers import SentenceTransformer


# Load
index = faiss.read_index("data/index.faiss")

with open("data/chunks.json") as f:
    data = json.load(f)

chunks = data["chunks"]
metadata = data["metadata"]

model = SentenceTransformer("all-MiniLM-L6-v2")


# 🔍 Retrieval
def retrieve(query, k=3):
    q_vec = model.encode([query])
    distances, indices = index.search(np.array(q_vec), k)

    results = [chunks[i] for i in indices[0]]
    sources = [metadata[i] for i in indices[0]]
    scores = distances[0]

    return results, sources, scores


# 🚨 Relevance Check
THRESHOLD = 1.2  # tune this

def is_relevant(scores):
    return min(scores) < THRESHOLD


# 🤖 Prompt
def build_prompt(context, query):
    return f"""
You are a strict assistant.

Answer ONLY from the provided context.
If the answer is not present, say:
"I don't have an answer for it"

Context:
{context}

Question:
{query}

Answer:
"""


# 🧠 LLM
def ask_llm(prompt):
    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "mistral",
            "prompt": prompt,
            "stream": False
        }
    )
    return res.json()["response"]


# 🔁 Final Pipeline
def chatbot(query):
    results, sources, scores = retrieve(query)

    if not is_relevant(scores):
        return "I don't have an answer for it"

    context = "\n\n".join(results)
    prompt = build_prompt(context, query)

    answer = ask_llm(prompt)

    return f"{answer}\n\nSources:\n" + "\n".join(set(sources))


# 🧪 CLI Loop
if __name__ == "__main__":
    while True:
        q = input("\nAsk: ")
        print(chatbot(q))


Ask:  who are you


I don't have an answer for it



Ask:  what is liquid


 Liquid is a template language created by Shopify, used to dynamically output objects and their properties and can further modify that output by creating logic with tags or directly altering it with filters. It's an open-source project available on GitHub and is used by many different software projects and companies.

Sources:
https://shopify.dev/docs/api/liquid



Ask:  how can I use liquid


...


Ask:  what is Angular


I don't have an answer for it



Ask:  what is the capital of india


I don't have an answer for it


KeyboardInterrupt: Interrupted by user

## Limitations

**🔹 Similarity (L2 Distance)**

Right it is using:
```python
faiss.IndexFlatL2()
```

👉 This measures Euclidean distance:
```
distance = √((x1 - y1)² + (x2 - y2)² + ...)
```
🧠 Interpretation:
- Smaller distance → more similar
- Larger distance → less similar

**❌ Problem 1: L2 is sensitive to magnitude:**
Even if two sentences mean the same, their embedding length (magnitude) can differ.

👉 L2 gets confused by:
- sentence length
- wording variation

**🔹 Chunking (Character-based)**

ATM, it is doing :
```python
text[i:i+500]
```

👉 This means:
- You are cutting text blindly every 500 characters
- No respect for: sentences, paragraphs, meaning boundaries

**❌ Problem 2: Bad chunking breaks meaning**

Example:
```
Liquid variables are used to store data. They can be output using {{ }} syntax.
```

Your chunks might become:
```
Chunk 1: "Liquid variables are used to store data. They can be out"
Chunk 2: "put using {{ }} syntax."
```

👉 Now:

- Meaning is split ❌
- Retrieval becomes weak ❌
- LLM gets incomplete context ❌